# Real-Time Analytics & Caching with Redis

## What you will learn in this lecture 🧐🧐

In this lecture, you’ll bring everything you’ve learned about Redis to life by building real-time analytics features in Python.
You’ll see how Redis enables live dashboards, query caching, and rolling metrics, which are essential for modern analytics engineers working with fast data pipelines.

By the end of this lecture, you’ll be able to:

* Implement real-time counters, rolling metrics, and leaderboards using Redis data types
* Cache SQL or API query results using cache-aside patterns
* Build simple real-time analytics dashboards connected to Redis
* Understand how Redis bridges streaming and analytical systems

## Why Real-Time Analytics Need Redis

Most traditional data warehouses like BigQuery, Snowflake, or PostgreSQL are built for batch processing.
They are excellent at computing aggregates over millions of rows — but they can’t easily answer questions like:

* “How many users are active right now?”
* “What are the top 5 products sold this minute?”
* “How many requests has our API processed in the last 10 seconds?”

Redis fills this gap. It holds the latest, most volatile data in memory, where it can be updated thousands of times per second, and read instantly by dashboards or APIs.

For analytics engineers, Redis acts as the real-time layer sitting between the data source and visualization tools.

## Connect Your Python Environment

You can reuse the free Redis Cloud instance you created earlier.


In [ ]:
import redis

HOSTNAME = "redis-10004.crce282.eu-west-3-1.ec2.cloud.redislabs.com"
PORT = 10004
PASSWORD = "xxxxx" # replace with your actual password

r = redis.Redis(
    host=HOSTNAME,
    port=PORT,
    password=PASSWORD,
    decode_responses=True
)

r.ping()  # Should return True

True

## Real-Time Counters

The simplest form of real-time analytics is **counting events** — page views, API calls, transactions, etc.

### Counting Page Views


In [17]:
r.incr("analytics:pageviews")  # +1 each time a page is visited
print(r.get("analytics:pageviews"))

16


### Counting Per Region or Time Bucket

In [31]:
from datetime import datetime, UTC
r.incr(f"analytics:pageviews:{datetime.now(UTC).strftime('%Y-%m-%d:%H:%M')}") # Run this multiple times

14

With the above code, you can basically reproduce a "group by".

In [32]:
keys = r.keys("analytics:pageviews:*")
for k in sorted(keys):
    print(k, r.get(k))


analytics:pageviews:2026-03-16:11:06 14


## Rolling Windows – Tracking Active Users in the Last 5 Minutes

Counting totals is simple, but most real dashboards need **rolling metrics**, like “active users in the last 5 minutes.”

We can do this with a **Sorted Set (ZSET)** using timestamps as scores.

### Add an event with timestamp


In [33]:
import time # For simulating last active users

now = time.time()
r.zadd("analytics:active_users", {f"user:{i}": now for i in range(10)})

10

You can now remove old users:

In [34]:
cutoff = now - 300  # 300 seconds = 5 minutes
r.zremrangebyscore("analytics:active_users", 0, cutoff)

0

Or count active users

In [35]:
active = r.zcard("analytics:active_users")
print(f"Active users in the last 5 minutes: {active}")

Active users in the last 5 minutes: 10


<Note type="hint" title="zcard">

`ZCARD` is a **Redis command** that returns the **number of elements** in a **sorted set**.
It stands for:

> **Z** = Sorted set, **CARD** = cardinality (i.e., count of elements)

### 🧱 Sorted sets recap

A **sorted set** (`ZSET`) in Redis is like a regular set, but every element has an **associated numeric score** that determines its **order**.

Think of it as:

```text
member → score
```

So you can efficiently do:

* ranking (`ZRANGE`, `ZREVRANGE`)
* leaderboards (`ZADD`, `ZINCRBY`)
* time-based ordering (`ZADD` with timestamps)

### ⚙️ Example

```python
r.zadd("leaderboard", {"luke": 100, "leia": 150, "vader": 200})
print(r.zcard("leaderboard"))   # → 3
```

Explanation:

* `ZADD` adds 3 members to the sorted set `leaderboard`.
* `ZCARD` returns **3**, the total number of elements.


### 📊 Analogy

| Type           | Example                        | Count command |
| -------------- | ------------------------------ | ------------- |
| **String**     | `"user:42" → "Leia"`           | *(n/a)*       |
| **List**       | `["a", "b", "c"]`              | `LLEN`        |
| **Set**        | `{a, b, c}`                    | `SCARD`       |
| **Hash**       | `{name: "Leia", tier: "gold"}` | `HLEN`        |
| **Sorted set** | `{(a,1), (b,2), (c,3)}`        | `ZCARD` ✅     |

Each collection type has its own “count” command:

* `LLEN` → list length
* `SCARD` → set size
* `HLEN` → hash field count
* `ZCARD` → sorted set size

</Note>

## Query Caching with the Cache-Aside Pattern

Analytics workloads often hit APIs or SQL databases for repeated queries.
Redis can **cache those results**, so you don’t need to recompute the same aggregates every time.

### The Cache-Aside Pattern

1. Check Redis for a cached result.
2. If it exists, return it.
3. If not, compute the value, store it in Redis, and return the result.


In [37]:
def get_avg_sales(region):
    key = f"cache:avg_sales:{region}"
    cached = r.get(key)
    if cached:
        print("✅ Cache hit")
        return float(cached)

    print("❌ Cache miss — computing...")
    # Simulate a slow SQL or API call
    time.sleep(2)
    avg_sales = 1523.7  # example computed result

    r.setex(key, 300, avg_sales)  # cache for 5 min
    return avg_sales

# Run twice to see the effect
get_avg_sales("eu")
get_avg_sales("eu")

✅ Cache hit
✅ Cache hit


1523.7

* ✅ First call → slow (cache miss)
* ✅ Second call → instant (cache hit)

<Note type="tip" title="Best use case">

Use the **cache-aside** pattern for **read-heavy** queries  
where data doesn’t change frequently (aggregations, summaries, KPIs).

</Note>

## Leaderboards – Ranking in Real Time

Redis Sorted Sets (`ZSETs`) make it easy to maintain live rankings.

### Add or Update Scores

In [38]:
r.zadd("leaderboard:sales", {"han": 1200, "luke": 1350, "leia": 1100})
r.zincrby("leaderboard:sales", 200, "leia")  # Leia makes another sale

1300.0

### Get Top Performers

In [39]:
top = r.zrevrange("leaderboard:sales", 0, 2, withscores=True)
print(top)
# [('luke', 1350.0), ('leia', 1300.0), ('han', 1200.0)]

[('luke', 1350.0), ('leia', 1300.0), ('han', 1200.0)]


Leaderboards aren’t just for games — they’re used in analytics for:

* Top-performing stores or sales agents
* Most viewed pages
* Best-selling products
* Most active users

<Note type="tip" title="Why use Redis for rankings">

Redis maintains sorted order automatically,  
so you never have to re-sort your leaderboard manually.

</Note>

## Event Streams – Capturing and Replaying Events

Redis **Streams** store events in time order and allow consumption in real time, much like Kafka but lighter.

### Add new events

In [40]:
r.xadd("events:clicks", {"user": "42", "page": "/home"})
r.xadd("events:clicks", {"user": "43", "page": "/pricing"})

'1773659638381-0'

In [41]:
### Read events
events = r.xrange("events:clicks", "-", "+", count=2)
for eid, data in events:
    print(eid, data)

1773659638367-0 {'user': '42', 'page': '/home'}
1773659638381-0 {'user': '43', 'page': '/pricing'}


In [42]:
### Consume continuously (new events only)
last_id = "0-0"
while True:
    new_events = r.xread({"events:clicks": last_id}, count=10, block=5000)
    if new_events:
        stream, messages = new_events[0]
        for msg_id, payload in messages:
            print("New event:", payload)
            last_id = msg_id
    
    break # Just to avoid an infinite loop in this example


New event: {'user': '42', 'page': '/home'}
New event: {'user': '43', 'page': '/pricing'}


This structure supports real-time analytics pipelines —
for example, clickstream ingestion or monitoring pipelines feeding live dashboards.

<Note type="tip" title="Stream use cases">
Streams are ideal for **real-time data ingestion**, **user activity tracking**, or **log processing** before sending data to your warehouse.
</Note>


## TTL & Automatic Cleanup

To prevent memory from growing indefinitely, always set TTLs for transient keys.


In [28]:
r.setex("analytics:active_users:eu", 60, 230)

True

You can also clean up   

In [43]:
for k in r.scan_iter("analytics:*"):
    if r.ttl(k) == -1:  # key has no expiration
        r.expire(k, 300)

This ensures Redis automatically expires old analytics data.

<Note type="tip" title="Memory hygiene">

Keeping TTLs consistent prevents unbounded memory usage  
and ensures Redis always contains **fresh, relevant data**.

</Note>

## Scaling Redis Analytics

When workloads grow:

* Use **TTLs** aggressively to keep the dataset small.
* **Shard** data by region or key prefix (`eu:`, `us:`, etc.).
* Stream historical data to a **warehouse or S3** for long-term analysis.
* Explore **RedisTimeSeries** or **Redis Streams** for time-series use cases.

In production, Redis often works alongside:

* PostgreSQL or BigQuery (storage layer)
* Kafka or Airflow (orchestration)
* Grafana or Streamlit (visualization)

Redis acts as the **real-time buffer** that connects streaming systems to analytical dashboards.

## Resources 📚📚

* [Redis for Real-Time Analytics (Redis University)](https://university.redis.com/)
* [Caching Patterns Explained (Redis Blog)](https://redis.io/learn/develop/java/cache/cache-strategies)
* [Redis Streams Documentation](https://redis.io/docs/latest/develop/data-types/streams/)
* [Streamlit Documentation](https://docs.streamlit.io/)
* [RedisTimeSeries Module](https://redis.io/docs/interact/time-series/)
* [Designing Data-Intensive Applications – Chapter 10 (Stream Processing)](https://dataintensive.net/)